# Southern Tree Species
Species
American Basswood
American Beech
American Elm
American sycamore 
Bald Cypress
Bigleaf Magnolia
Bitternut Hickory
Black Cherry
Black locust 
Black Oak 
Black walnut 
Black Willow
Blackgum 
Boxelder
Cherrybark Oak
Chestnut Oak
Chinkapin oak 
Common persimmon 
Eastern cottonwood 
Eastern Hophornbeam
Eastern Redbud
Eastern Redcedar
Eastern White Pine
Florida Maple
Flowering Dogwood
Green Ash
Hackberry
Honeylocust
Live Oak
Loblolly Pine
Longleaf Pine
Mimosa
Mockernut Hickory
Musclewood
Northern Red Oak
Ohio Buckeye
Osage-orange
Overcup Oak
Pecan
Pignut Hickory
Pin Oak 
Post Oak
Red Maple
River Birch
Sassafras
Sawtooth Oak
Scarlet Oak
Serviceberry
Shagbark hickory
shingle oak
Shortleaf Pine
silver maple
Slash Pine
slippery elm
Sourwood
Southern Magnolia
Southern Red Oak
sugar maple
Sugarberry
Swamp Chestnut Oak
Sweetbay Magnolia
sweetgum
Virginia Pine
Water Oak
White Ash
White Oak
Willow Oak
Winged Elm
Yellow Poplar

The following code block imports the labels CSV as a Pandas dataframe. It then displays the first five rows of the dataframe and prints the original row count.

The code filters out rows where the species was listed as "other" and prints the remaining row count.

Afterward, the Species column is selected as the target label, while the remaining columns are stored separately as input features. These inputs and labels are then split into training and testing sets. For initial exploration, only 5% of the data is used for training so that model runs stay fast

In [86]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('bark_images_and_labels/bark_class_labels.csv')
display(df.head())
print("Original dataframe:", df.shape[0])

df = df[~df['Species'].str.lower().eq('other')].copy()
print("Dataframe after removing 'other' species:", df.shape[0])

print("Number of unique species:", df['Species'].nunique())

sample_tree_df, remaining_tree_df = train_test_split(
    df, 
    test_size=0.95, 
    random_state=42,
    stratify=df["Species"]
)
print("Training set size:", sample_tree_df.shape[0])
print("Number of unique species in training set:", sample_tree_df['Species'].nunique())

print("Remaining set size:", remaining_tree_df.shape[0])
print("Number of unique species in remaining set:", remaining_tree_df['Species'].nunique())

,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,Photo0,Photo1,Photo2,Photo3
0,1,Pin oak,Dry,17.0,NaN,tdb96463_USG,2024/02/05 18:42:18.856+00,tdb96463_USG,2024/02/05 18:42:18.856+00,ATT1_Photo 3.jpg,ATT2_Photo 2.jpg,ATT3_Photo 4.jpg,ATT4_Photo 1.jpg
1,2,Eastern white pine,Dry,29.0,NaN,tdb96463_USG,2024/02/05 18:49:21.604+00,tdb96463_USG,2024/02/05 18:49:21.604+00,ATT5_Photo 3.jpg,ATT6_Photo 2.jpg,ATT7_Photo 4.jpg,ATT8_Photo 1.jpg
2,3,Yellow-poplar,Dry,18.0,NaN,tdb96463_USG,2024/02/05 18:50:52.876+00,tdb96463_USG,2024/02/05 18:50:52.876+00,ATT9_Photo 2.jpg,ATT10_Photo 3.jpg,ATT11_Photo 1.jpg,ATT12_Photo 4.jpg
3,4,Loblolly pine,Dry,16.0,NaN,tdb96463_USG,2024/03/08 20:53:15.938+00,tdb96463_USG,2024/03/08 20:53:15.938+00,ATT13_Photo 2.jpg,ATT14_Photo 3.jpg,ATT15_Photo 4.jpg,ATT16_Photo 1.jpg
4,5,Loblolly pine,Dry,17.0,NaN,tdb96463_USG,2024/03/08 20:56:20.335+00,tdb96463_USG,2024/03/08 20:56:20.335+00,ATT17_Photo 1.jpg,ATT18_Photo 4.jpg,ATT19_Photo 3.jpg,ATT20_Photo 2.jpg


Original dataframe: 692
Dataframe after removing 'other' species: 685
Number of unique species: 25
Training set size: 34
Number of unique species in training set: 21
Remaining set size: 651
Number of unique species in remaining set: 25


The following code block melts the sampled tree-level dataframe across the four photo columns. This converts the data from one row per tree, with separate Photo0, Photo1, Photo2, and Photo3 columns, into one row per image with a single Image column.

The OBJECTID and Species columns are preserved so that each melted image row still keeps its tree identifier and species label. Rows with missing image filenames are removed, and the index is reset.

The code then builds a full ImagePath column by combining the extracted image folder path with each filename in the Image column. It verifies that all images for tree 598 remain together in the sampled dataframe, displays the sample image count and first few rows, and checks whether the generated image paths exist on disk.

In [87]:
import os

photo_labels = ["Photo0", "Photo1", "Photo2", "Photo3"]
extract_path = 'bark_images_and_labels/BARK_UGA_PICS/'

def melt_photos(split_df):
    melted_df = split_df.melt(
        id_vars=['OBJECTID', 'Species'], 
        value_vars=photo_labels, 
        var_name='Photo', 
        value_name='Image'
    )
    
    return melted_df.dropna(subset=['Image']).reset_index(drop=True)

def build_image_paths(image_df, extract_path):
    image_df['ImagePath'] = extract_path + image_df['Image']
    return image_df

sample_image_df = melt_photos(sample_tree_df)
sample_image_df = build_image_paths(sample_image_df, extract_path)
path_exists_df = sample_image_df["ImagePath"].apply(os.path.exists)

print("Verification of tree level split using tree number 598")
display(sample_image_df[sample_image_df["OBJECTID"] == 598])

print("Sample set image count:", sample_image_df.shape[0])
display(sample_image_df.head())

display(path_exists_df.head())
print(path_exists_df.value_counts())

Verification of tree level split using tree number 598


,OBJECTID,Species,Photo,Image,ImagePath
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2385_P...
37,598,Water oak,Photo1,ATT2386_Photo 4.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2386_P...
71,598,Water oak,Photo2,ATT2387_Photo 1.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2387_P...
105,598,Water oak,Photo3,ATT2388_Photo 3.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2388_P...


Sample set image count: 136


,OBJECTID,Species,Photo,Image,ImagePath
0,40,Southern magnolia,Photo0,ATT157_Photo 4.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT157_Ph...
1,54,Longleaf pine,Photo0,ATT213_Photo 2.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT213_Ph...
2,615,Eastern white pine,Photo0,ATT2453_Photo 4.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2453_P...
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT2385_P...
4,203,Willow oak,Photo0,ATT809_Photo 3.jpg,bark_images_and_labels/BARK_UGA_PICS/ATT809_Ph...


0    True
1    True
2    True
3    True
4    True
Name: ImagePath, dtype: bool

ImagePath
True    136
Name: count, dtype: int64


The following code block defines the folder path where the extracted bark images are stored. It then defines a function that creates a new ImagePath column by combining the base image folder path with each image filename from the Image column.

The function is applied to both the training and testing image dataframes. Finally, the first few rows of the training dataframe are displayed to verify that the image paths were created correctly.

In [88]:
import torch
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import gc

class BarkDataset(Dataset):
        def __init__(self, image_df, transform=None):
            self.image_df = image_df
            self.transform = transform
        
        def __len__(self):
            return len(self.image_df)
        
        def __getitem__(self, index):
            row = self.image_df.iloc[index]
            image_path = row["ImagePath"]
            label = row["Label"]

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as e:
                raise RuntimeError(f"Error loading image at {image_path}") from e

            
            if self.transform is not None:
                image = self.transform(image)

            label = torch.tensor(label, dtype=torch.long)
            
            return image, label

class TrainTestBarkModel():
    def __init__(self, image_df):
        self.image_df = image_df.reset_index(drop=True).copy()

        self.image_df['Label'] = LabelEncoder().fit_transform(image_df['Species'])

        self.num_classes = self.image_df['Label'].nunique()

        self.label_map_df = (
            self.image_df[["Label", "Species"]]
            .drop_duplicates()
            .sort_values("Label")
            .reset_index(drop=True)
        )
    
    def show_label_map(self):
        print("Label to Species mapping: ", self.num_classes, " classes")
        display(self.label_map_df)

    def get_num_classes(self):
        return self.num_classes
        
    def get_num_images(self):
        return len(self.image_df)

    def get_dataloader(self, image_size_tuple, batch_size, shuffle):
        train_transform = transforms.Compose([
            transforms.Resize(image_size_tuple),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

        eval_transform = transforms.Compose([
            transforms.Resize(image_size_tuple),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        dataset = BarkDataset(
            self.image_df, 
            transform=train_transform
        )

        dataloader = DataLoader(
            dataset, 
            batch_size=batch_size, 
            shuffle=shuffle
        )

        return dataloader
    
    def train_model(self, model_tuple, batch_size, num_epochs):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if num_epochs == None:
            num_epochs = self.get_num_images() // batch_size

        model = model_tuple[1](weights="DEFAULT")
        
        if hasattr(model, "fc"):
            model.fc = nn.Linear(
                model.fc.in_features, 
                self.num_classes
            )
        elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
            model.classifier[-1] = nn.Linear(
                model.classifier[-1].in_features,
                self.num_classes
            )
        else:
            raise ValueError("Unsupported model architecture")

        model = model.to(device)
        print(f"\n===== Training {model_tuple[0]} =====")
        print(f"Model loaded on device: {device}")
        print(f"Number of classes: {self.num_classes}")

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        for epoch in range(num_epochs):
            model.train()

            running_loss = 0.0
            correct = 0
            total = 0

            for images, labels in self.get_dataloader(image_size_tuple=model_tuple[2], batch_size=batch_size, shuffle=True):
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()

                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                predicted_labels = outputs.argmax(dim=1)
                correct += (predicted_labels == labels).sum().item()
                total += labels.size(0)
        
            epoch_loss = running_loss / total
            epoch_accuracy = correct / total

            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"Loss: {epoch_loss:.4f}")
            print(f"Accuracy: {epoch_accuracy:.4f}")
            print()

    def train_multiple(self, models, batch_size, num_epochs):
        results = []

        for model_tuple in models:
            try:
                self.train_model(
                    model_tuple=model_tuple,
                    batch_size=batch_size,
                    num_epochs=num_epochs,
                )
                results.append({"model": model_tuple[0], "status": "passed", "error": None})
            except Exception as e:
                print(f"Smoke test failed for {model_tuple[0]}: {e}")
                results.append({"model": model_tuple[0], "status": "failed", "error": str(e)})
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        smoke_results_df = pd.DataFrame(results)
        display(smoke_results_df)

In [ ]:
from torchvision.models import (
    # ResNet models
    resnet18,
    resnet34,
    resnet50,
    resnet101,
    resnet152,

    # EfficientNet B models
    efficientnet_b0,
    efficientnet_b1,
    efficientnet_b2,
    efficientnet_b3,
    efficientnet_b4,
    efficientnet_b5,
    efficientnet_b6,
    efficientnet_b7,

    # EfficientNet V2 models
    efficientnet_v2_s,
    efficientnet_v2_m,
    efficientnet_v2_l,
)

models = [
    # ResNet models
    ("resnet18", resnet18, (224, 224)),
    ("resnet34", resnet34, (224, 224)),
    ("resnet50", resnet50, (224, 224)),
    ("resnet101", resnet101, (224, 224)),
    ("resnet152", resnet152, (224, 224)),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, (224, 224)),
    ("efficientnet_b1", efficientnet_b1, (240, 240)),
    ("efficientnet_b2", efficientnet_b2, (260, 260)),
    ("efficientnet_b3", efficientnet_b3, (300, 300)),
    ("efficientnet_b4", efficientnet_b4, (380, 380)),
    ("efficientnet_b5", efficientnet_b5, (456, 456)),
    ("efficientnet_b6", efficientnet_b6, (528, 528)),
    ("efficientnet_b7", efficientnet_b7, (600, 600)),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, (384, 384)),
    ("efficientnet_v2_m", efficientnet_v2_m, (480, 480)),
    ("efficientnet_v2_l", efficientnet_v2_l, (480, 480)),
]

MODEL_OPTION = 0
BATCH_SIZE = 16
NUM_EPOCHS = None

bulk_trainer = TrainTestBarkModel(sample_image_df)

if MODEL_OPTION == 0:
    bulk_trainer.train_model(model_tuple=("resnet18", resnet18, (224, 224)), batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 1:
    bulk_trainer.train_multiple(models=models, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)


===== Training resnet18 =====
Model loaded on device: cuda
Number of classes: 21
Epoch 1/136
Loss: 3.3356
Accuracy: 0.2059

Epoch 2/136
Loss: 3.0860
Accuracy: 0.1544

Epoch 3/136
Loss: 3.0858
Accuracy: 0.2132

Epoch 4/136
Loss: 3.0708
Accuracy: 0.1912

Epoch 5/136
Loss: 3.0371
Accuracy: 0.1838

Epoch 6/136
Loss: 3.0157
Accuracy: 0.2206

Epoch 7/136
Loss: 2.9701
Accuracy: 0.2353

Epoch 8/136
Loss: 3.0092
Accuracy: 0.1985

Epoch 9/136
Loss: 2.9676
Accuracy: 0.2353

Epoch 10/136
Loss: 2.9756
Accuracy: 0.2353

Epoch 11/136
Loss: 2.9538
Accuracy: 0.2353

Epoch 12/136
Loss: 2.9432
Accuracy: 0.2206

Epoch 13/136
Loss: 2.9683
Accuracy: 0.2132

Epoch 14/136
Loss: 2.9991
Accuracy: 0.2206

Epoch 15/136
Loss: 3.0170
Accuracy: 0.2059

Epoch 16/136
Loss: 3.0177
Accuracy: 0.1838

Epoch 17/136
Loss: 2.9768
Accuracy: 0.2206

Epoch 18/136
Loss: 3.0009
Accuracy: 0.2206

Epoch 19/136
Loss: 2.9803
Accuracy: 0.2353

Epoch 20/136
Loss: 2.9717
Accuracy: 0.2132

Epoch 21/136
Loss: 2.9304
Accuracy: 0.2132

Epo

KeyboardInterrupt: 